In [1]:
from torch.utils.data import DataLoader, TensorDataset

from gensim.models import Word2Vec
from model         import GenerateModel
from metrics       import eval_model, compare_metric
from evaluation      import all_metrics
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy             as np
import torch.nn            as nn
from tqdm import tqdm
import copy
import random
import torch
import os

torch.manual_seed(42)
random.seed(42)
# device     = torch.device('mps') if torch.backends.mps.is_available() else torch.device('cpu'); print(f'Deivce: {device}')
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu'); print(f'Deivce: {device}')
model_path = os.path.join(os.getcwd(),'Model','model.tar')

Deivce: cuda


In [2]:
X_test  = np.load('X_test.npy')
Y_test  = np.load('Y_test.npy')

X_test  = torch.from_numpy(X_test) .long()
Y_test  = torch.from_numpy(Y_test) .type(torch.float32) 

test_loader  = DataLoader(TensorDataset(X_test ,Y_test) , batch_size = 64,shuffle=False)

In [3]:
@torch.no_grad()
def eval_model(model, device, data_loader,label_space):
    model.eval()
    all_pred     = torch.empty((0,), dtype = torch.float32).to(device)
    all_labels   = torch.empty((0,), dtype = torch.float32).to(device)
    all_pred_raw = torch.empty((0,), dtype = torch.float32).to(device)

    for data_inputs, data_labels in data_loader:
        data_inputs = data_inputs.to(device)
        data_labels = data_labels.to(device)
        preds, _    = model(data_inputs)
        pred_labels = (F.sigmoid(preds) >= 0.5).long()

        all_pred     = torch.cat((all_pred,pred_labels), dim = 0 )
        all_labels   = torch.cat((all_labels,data_labels), dim = 0)
        all_pred_raw = torch.cat([all_pred_raw, preds], dim = 0)

    #print(classification_report(y_pred =   all_pred.cpu().numpy(), y_true = all_labels.cpu().numpy()))
    return (all_metrics(yhat =   all_pred.cpu().numpy(), y = all_labels.cpu().numpy(), yhat_raw=all_pred_raw.cpu().numpy()  ))

In [4]:
def load_client_data(client_idx: int, num_clients: int, idxs) -> TensorDataset:
    X = np.load('X_train.npy')
    Y = np.load('Y_train.npy')

    X = torch.from_numpy(X).long()
    Y = torch.from_numpy(Y).type(torch.float32)

    indices = [idx for idx, val in enumerate(idxs) if client_idx == val]
    X_client = X[indices]
    y_client = Y[indices]

    return TensorDataset(X_client, y_client)

def client_update(model  : nn.Module, train_loader: DataLoader, epochs : int = 1,lr: float = 0.0001, device : str = 'cpu') -> dict:
    model.to(device)
    model.train()
    # optimizer = optim.SGD(model.parameters(), lr=lr)
    optimizer   = torch.optim.Adam(model.parameters(), lr = 0.001, betas =  (0.9,0.99)) #(0.1,0.3)
    loss_fn = nn.CrossEntropyLoss()

    for _ in range(epochs):
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            preds, alpha = model(X_batch)
            loss = loss_fn(preds, y_batch)
            loss.backward()
            optimizer.step()

    return copy.deepcopy(model.state_dict())

def server_aggregate(global_model: nn.Module, client_state_dicts: list) -> nn.Module:
    global_dict = global_model.state_dict()
    for key in global_dict.keys():
        stacked = torch.stack([client_dict[key].float() for client_dict in client_state_dicts], dim=0)
        global_dict[key] = torch.mean(stacked, dim=0)
    global_model.load_state_dict(global_dict)
    return global_model

In [ ]:
num_clients  = 3
rounds       = 2500
local_epochs = 4
batch_size   = 64

model_path   = os.path.join(os.path.join(os.getcwd(),'Model'),'federated_model.tar')
global_model = GenerateModel()
global_model.load_state_dict(torch.load(model_path, weights_only=True))

history = []

X = np.load('X_train.npy')
total_samples = X.shape[0]
data_points = list(random.choices(population=[0,1,2],k = total_samples))

client_loaders = []
for client_idx in range(num_clients):
    dataset = load_client_data(client_idx, num_clients, idxs=data_points)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    client_loaders.append(loader)

for rnd in tqdm(range(rounds)):
    client_state_dicts = []
    for client_idx, loader in enumerate(client_loaders, 1):
        client_model  = GenerateModel()
        client_model.load_state_dict(global_model.state_dict())
        updated_state = client_update( client_model,loader,epochs=local_epochs,device = device)
        client_state_dicts.append(updated_state)
    global_model = server_aggregate(global_model, client_state_dicts)
    global_model.to(device)
    history.append(eval_model(global_model,device,test_loader,label_space=50))

In [ ]:
plt.figure(figsize=(7,7))
for key in history[0].keys():
    plt.plot([m[key] for m in history], label = key)
plt.legend()
plt.savefig('Federated_Training.png') 

---

In [ ]:
# model_path2 = os.path.join(os.path.join(os.getcwd(),'Model'),'federated_model4.tar')
# torch.save(global_model.state_dict(),f = model_path2) 

In [5]:
model_path2 = os.path.join(os.path.join(os.getcwd(),'Model'),'federated_model3.tar')
global_model = GenerateModel()
global_model.load_state_dict(torch.load(model_path2, weights_only=True))
global_model.to(device)
global_model.eval()

federated_test = \
       eval_model(model = global_model,
           device       = device,
           label_space  = 50,
           data_loader  = test_loader)

benchmark = {
    'auc_macro' : 0.884,
    'auc_micro' : 0.916,
    'f1_macro'  : 0.576,
    'f1_micro'  : 0.633
}

# Comparing the history of a limited model
for key in benchmark.keys():
    if key in federated_test:
        print(compare_metric(benchmark,federated_test,key),'<br>')

auc_macro: 2.68% change (0.8840 → 0.9077) <br>
auc_micro: 1.78% change (0.9160 → 0.9323) <br>
f1_macro: 7.66% change (0.5760 → 0.6202) <br>
f1_micro: 2.42% change (0.6330 → 0.6483) <br>
